# Progetto DIA - A.A 2025/26

Autori: Justin Carideo (justin.carideo@studio.unibo.it), Laura Bertozzi (laura.bertozzi5@studio.unibo.it)

# Setup

## Caricamento librerie

* `NumPy` per creare e operare su array a n dimensioni
* `mathplotlib` e `seaborn` per creare grafici
* `pandas` per creare e manipolare dati tabulari
* `scikit-learn` per la gestione dei modelli di apprendimento
* `os` e `kaggle` per il download dei dataset


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import sklearn
import seaborn as sns
import kaggle
import os
import os.path

sns.set_theme(style="whitegrid")

In [ ]:

# Librerie scikit-learn
from sklearn.model_selection import KFold, StratifiedKFold, RandomizedSearchCV
from sklearn.tree import plot_tree
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.metrics import accuracy_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression

from xgboost import XGBClassifier


Importiamo i dataset presi dal sito https://www.kaggle.com. Questi due dataset fanno riferimento a dati in ambito agricolo, in particolare:
1) **Crop Dataset** -> dataset che presenta variabili legate a caratteristiche del terreno e dei nutrienti presenti, utili a dedurre quale sia la coltura ideale (https://www.kaggle.com/datasets/varshitanalluri/crop-recommendation-dataset)
2) **Fertilizer DataSet** -> dataset che permettte di individuare il tipo di fertilizzante più efficace dato il tipo di terreno e la coltura piantata. (https://www.kaggle.com/datasets/nishchalchandel/fertilizer-recommendation)

In [ ]:
CROP_DATASET_ID = "varshitanalluri/crop-recommendation-dataset"
FERTILIZER_DATASET_ID = "nishchalchandel/fertilizer-recommendation"

BASE_DOWNLOAD_DIR = "data"

CROP_DATASET_DIR = os.path.join(BASE_DOWNLOAD_DIR, "dataset_1")
FERTILIZER_DATASET_DIR = os.path.join(BASE_DOWNLOAD_DIR, "dataset_2")

os.makedirs(BASE_DOWNLOAD_DIR, exist_ok=True) # crea la directory se non esiste


def download_kaggle_dataset(dataset_id, dataset_path):
    os.makedirs(dataset_path, exist_ok=True)
    if not os.listdir(dataset_path):
        print(f"Download {dataset_id} in {dataset_path}")
        kaggle.api.dataset_download_files(
            dataset_id,
            path=dataset_path,
            unzip=True
        )
    else:
        print(f"Dataset already in specified path")

download_kaggle_dataset(CROP_DATASET_ID, CROP_DATASET_DIR)
download_kaggle_dataset(FERTILIZER_DATASET_ID, FERTILIZER_DATASET_DIR)

CROP_DATASET_PATH = os.path.join(CROP_DATASET_DIR, "Crop_recommendation.csv")
FERTILIZER_DATASET_PATH = os.path.join(FERTILIZER_DATASET_DIR, "fertilizer_recommendation_dataset.csv")

Creiamo i dataframe di entrambi i dataset:

In [ ]:
crop_df = pd.read_csv(CROP_DATASET_PATH)
fertilizer_df = pd.read_csv(FERTILIZER_DATASET_PATH)

# Analisi Esplorativa

### Colonne di crop_df

* **Nitrogen** la proporzione di azoto nel terreno
* **Phosohorus** la proporzione di fosforo nel terreno
* **Potassium** la proporzione di potassio nel terreno
* **Temperature** la temperatura in gradi Celsius
* **Humidity** unidità relativa in percentuale
* **pH_Value** valore pH del terreno
* **Rainfall** precipitazioni in mm
* **Crop** la feature da predire: la coltura ideale

In [ ]:
crop_df.info()

### Colonne di fertilizer_df

* **Temperature** la temperatura in gradi Celsius
* **Moisture** l'umidità del terreno in decimale
* **Rainfall** precipitazioni in mm
* **PH** valore pH del terreno
* **Nitrogen** azoto contenuto nel terreno
* **Phosphorous** fosforo contenuto nel terreno
* **Potassium** potassio contenuto nel terreno
* **Carbon** carbone organico contenuto nel terreno
* **Soil** tipologia di terreno
* **Crop** coltura piantata
* **Fertilizer** la feature da predire: il fertilizzante ottimale
* **Remark** eventuali osservazioni

In [ ]:
fertilizer_df.info()

In [ ]:
crop_df.describe()

In [ ]:
fertilizer_df.describe()

Osserviamo ora quali colture vengono prese in considerazione dai dataset.

Poniamo quindi le stringhe dei nomi delle colture in minuscolo ed eliminiamo eventuali spazi, per poi cercare corrispondenze tra di esse.

In [ ]:
crops_fertilizer = fertilizer_df["Crop"].unique()
crops_fertilizer = [x.lower().replace(" ", "") for x in crops_fertilizer]
print(crops_fertilizer)

In [ ]:
crops_crop = crop_df["Crop"].unique()
crops_crop = [x.lower().replace(" ", "") for x in crops_crop]
print(crops_crop)

In [ ]:
common = list(set(crops_fertilizer) & set(crops_crop))
print("Common elements:", common)
difference_fert = list(set(crops_crop) - set(common))
print("\nElements in crop but not in fertilizer:", difference_fert)
difference_crop = list(set(crops_fertilizer) - set(common))
print("Elements in fertilizer but not in crop:", difference_crop)


print("\nNumber of common elements:", len(common))
print("Number of elements in crop:", len(crops_crop))

Notiamo che le colture di `crop_df` rientrano perfettamente nelle colture di `fertilizer_df`. Di conseguenza possiamo portarle allo stesso formato.

In [ ]:
def clean_crop_name(name):
    return str(name).lower().replace(" ", "")

In [ ]:
crop_df["Crop"] = crop_df["Crop"].map(clean_crop_name)

In [ ]:
fertilizer_df["Crop"] = fertilizer_df["Crop"].map(clean_crop_name)

Ananalizziamo ora le altre variabili prese in considerazione dai due dataset, ponendo particolare attenzione su quelle in comune.

In [ ]:
crop_keys = set(crop_df.keys())
fertilizer_keys = set(fertilizer_df.keys())


print("variabili di \"crop_df\":", list(crop_keys))
print("numero di variabili di \"crop_df\":", len(crop_keys))
print("\nvariabili di \"fertilizer_df\":", list(fertilizer_keys))
print("numero di variabili di \"fertilizer_df\":", len(fertilizer_keys))

common_keys = list(set(crop_keys) & set(fertilizer_keys))
print("\nvariabili comuni:", common_keys)
print("numero di variabili comuni:", len(common_keys))

different_a = list(set(fertilizer_keys) - set(common_keys))
different_b = list(set(crop_keys) - set(common_keys))
different_keys = list(set(crop_keys) ^ set(fertilizer_keys))
print("\nvariabili diverse:", different_keys)
print("numero di variabili diverse:", len(different_keys))

Tra le variabili diverse notiamo che tre di esse differiscono solo nel nome, ossia:
* Humidiy e Moisture
* PH e pH_Value
* Phosphorous e Phosphorus

Le restanti quattro rappresentano parametri effettivamente differenti:
* Soil
* Carbon
* Fertilizer (incognita del fertilizer dataset)
* Remark (che ignoriamo per il momento)

Notiamo come alcuni parametri non facciano riferimento alla stessa scala(i.e. humidity -> in percentuale (72%), Moisture -> in proporzione (0.72)). In compenso, per quanto riguarda il resto:
* Temperature -> °C
* Nitrogen, Phosphorous, Carbon, Potassium -> mg/kg
* pH -> misura adimensionale standard (da 0 a 14)
* Rainfall -> mm

In [ ]:
map_names = {
    "Moisture": "Humidity",
    "PH" : "pH_Value",
    "Phosphorous" : "Phosphorus"
}

fertilizer_df = fertilizer_df.rename(columns=map_names)
fertilizer_df["Humidity"] = fertilizer_df["Humidity"] * 100 # Per evitare problemi con la divisione

In questo modo abbiamo portato entrambe le misurazioni di umidità in percentuale e mappato i parametri ad un nome comune. Ora possiamo creare un dataset unificato.

In [ ]:
column_names = ["Temperature", "Humidity", "Rainfall", "pH_Value", "Nitrogen", "Phosphorus", "Potassium", "Crop"]

merged_df = pd.concat([
    fertilizer_df[column_names],
    crop_df[column_names]],
    axis=0,
    join='outer',
    ignore_index=True
)

merged_df.describe()

### Osservazione dei campioni mediante diagrammi



Distribuzione delle Colture:

In [ ]:
plt.figure(figsize=(14, 6))
sns.countplot(data=crop_df, x='Crop', hue='Crop', palette='viridis', order=crop_df["Crop"].value_counts().index)
plt.title("Distribuzione Colture (crop_df)")
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(14, 6))
sns.countplot(data=fertilizer_df, x='Crop', hue='Crop', palette='viridis', order=fertilizer_df["Crop"].value_counts().index)
plt.title("Distribuzione Colture (fertilizer_df)")
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

Notiamo come entrambi i dataset presentino 100 campioni per coltura.
Per crop_df quindi non vi sono sbilanciamenti tra classi.

In [ ]:
plt.figure(figsize=(14, 6))
sns.countplot(data=fertilizer_df, x='Fertilizer', hue='Fertilizer', palette='magma', order=fertilizer_df["Fertilizer"].value_counts().index)
plt.title("Distribuzione Fertilizzanti (fertilizer_df)")
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

Per i fertilizzanti invece le classi sono fortemente sbilanciate.

Definiamo una funzione per fare il plot dei parametri chimici che andremmo ad analizzare

In [ ]:
def plot_chemicals(chemical):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    sns.histplot(crop_df[chemical], kde=True, color='blue', ax=axes[0])
    axes[0].set_title(chemical + ' crop_df', fontsize=12)
    axes[0].set_xlabel(chemical)

    sns.histplot(fertilizer_df[chemical], kde=True, color='red', ax=axes[1])
    axes[1].set_title(chemical + ' fertilizer_df', fontsize=12)
    axes[1].set_xlabel(chemical)

Valori di umidità rilevati:

In [ ]:
plot_chemicals('Humidity')

Temperature misurate:

In [ ]:
plot_chemicals('Temperature')

Precipitazioni misurate: 

In [ ]:
plot_chemicals('Rainfall')

In [ ]:
plot_chemicals('Nitrogen')

In [ ]:
plot_chemicals('Phosphorus')

In [ ]:
plot_chemicals('pH_Value')

In [ ]:
plot_chemicals('Potassium')

Notiamo un'incoerenza nel dataset `fertilizer_df`, in quanto i parametri `Rainfall`, `Potassium`, `Carbon` e `Phosphorus` misurano valori negativi. Essendo la misurazione delle precipitazioni in mm, non è possibile che si presentino valori negativi. Per tanto abbiamo deciso di portare tali valori a zero.

In [ ]:
print((fertilizer_df[['Potassium', 'Carbon', 'Rainfall', 'Phosphorus']] < 0).sum())

In [ ]:
fertilizer_df['Potassium'] = fertilizer_df['Potassium'].clip(lower=0)
fertilizer_df['Carbon'] = fertilizer_df['Carbon'].clip(lower=0)
fertilizer_df['Rainfall'] = fertilizer_df['Rainfall'].clip(lower=0)
fertilizer_df['Phosphorus'] = fertilizer_df['Phosphorus'].clip(lower=0)

Questa scelta è stata presa in assenza di documentazione affidabile sulle misurazioni, dando per scontato che i valori inseriti come negativi fossero in realtà vicini allo zero.

### Correlazione tra variabili

Analizziamo la correlazione tra umidità e precipitazioni mediante coefficiente di Pearsonn.

In [ ]:
rice_crop = crop_df[crop_df["Crop"] == "rice"]
rice_fertilizer = fertilizer_df[fertilizer_df["Crop"] == "rice"]
hum_rain_corr_crop = rice_crop["Humidity"].corr(rice_crop["Rainfall"])
hum_rain_corr_fertilizer = rice_fertilizer["Humidity"].corr(rice_fertilizer["Rainfall"])

print(f"Correlation between Humidity and Rainfall in crop_df: {hum_rain_corr_crop:.2f}")
print(f"Correlation between Humidity and Rainfall in fertilizer_df: {hum_rain_corr_fertilizer:.2f}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.scatterplot(x = rice_crop["Humidity"], y = rice_crop["Rainfall"], color='blue', ax=axes[0])
axes[0].set_title('Humidity vs Rainfall (crop_df)', fontsize=12)
axes[0].set_xlabel('Humidity (%)')
axes[0].set_ylabel('Rainfall (mm)')

sns.scatterplot(x = rice_fertilizer["Humidity"], y = rice_fertilizer["Rainfall"], color='red', ax=axes[1])
axes[1].set_title('Humidity vs Rainfall (fertilizer_df)', fontsize=12)
axes[1].set_xlabel('Humidity (%)')
axes[1].set_ylabel('Rainfall (mm)')


Notiamo come non vi sia correlazione tra le due variabili, probabilmente perchè il range dell'umidità è ristretto e perchè la relazione non è lineare. Tentiamo ora la stessa analisi con il coefficiente di Spearman.

In [ ]:
rice_crop = crop_df[crop_df["Crop"] == "rice"]
rice_fertilizer = fertilizer_df[fertilizer_df["Crop"] == "rice"]
hum_rain_corr_crop = rice_crop["Humidity"].corr(rice_crop["Rainfall"], method='spearman')
hum_rain_corr_fertilizer = rice_fertilizer["Humidity"].corr(rice_fertilizer["Rainfall"], method='spearman')

print(f"Correlation between Humidity and Rainfall in crop_df: {hum_rain_corr_crop:.2f}")
print(f"Correlation between Humidity and Rainfall in fertilizer_df: {hum_rain_corr_fertilizer:.2f}")

Il risultato è pressochè identico, confermando che tra le due variabili Humidity/Moisture e Rainfall non c'è correlazione.

### Valutazione media e quartili

Valutiamo ora, per ogni coltura, i valori tipici di azoto, potassio, fosforo e temperatura.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(20, 20))
num_cols = ["Nitrogen", "Potassium", "Phosphorus", "Temperature"]

for ax, col in zip(axes.flatten(), num_cols):
    sns.boxplot(data=crop_df, x="Crop", y=col, ax=ax)
    ax.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(20, 20))
num_cols = ["Nitrogen", "Potassium", "Phosphorus", "Temperature"]

for ax, col in zip(axes.flatten(), num_cols):
    sns.boxplot(data=fertilizer_df, x="Fertilizer", y=col, ax=ax)
    ax.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
fertilizer_df[["Fertilizer", "Phosphorus"]].sort_values("Phosphorus", ascending=False).head(3)

In [ ]:
print((fertilizer_df['Phosphorus'] > 177).sum())
fertilizer_df['Phosphorus'].describe()

Sono presenti numerosi outlier nel dataser Fertilizer_df.
Considerando ad esempio la colonna Phosforus, ben 76 valori risultano pari al massimo: si tratta di un risultato che fa sospettare la generazione di valori di fill, ipotesi rafforzata inoltre dalla presenza di valori negativi.

Proseguiamo, simulando che si tratti di valori effettivamente misurati.

# Addestramento dei modelli
Una volta estratte le feature più importanti possiamo passare al tipo di modello che vogliamo addestrare.

## Modello 1
Il primo modello sarà un modello di *classificazione* per la previsione del parametro `Crop` attraverso degli **alberi di classificazione**, in particolare alleniamo due modelli:
- uno che userà `RandomForestClassifier`, che sfrutta l'algoritmo *Random Forest*
- uno che userà `xgboost`, che sfrutta invece *Gradient Boosting*

In [ ]:
y_merged = merged_df['Crop']
X_merged = merged_df.drop('Crop', axis=1)

Definiamo una funzione generale utile per i classificatori. Attualmente stiamo analizzando dati *numerici*, quindi non abbiamo bisogno di One-Hot Encoding o tecniche complesse per elaborare i dati. L'unico strumento di preprocessing incluso è quindi`StandardScaler`.

In [ ]:
def build_scaled_classificator(classifier):
    return Pipeline(steps=[
        ("scaler", StandardScaler()),
        ("classifier", classifier)
    ])

### Random Forest

In [ ]:
X_train_merged, X_test_merged, y_train_merged, y_test_merged = train_test_split(X_merged, y_merged, test_size=0.30, random_state=42)

In [ ]:
# griglia di parametri per Random Forest
param_grid = {
    'classifier__n_estimators': [50, 100, 200, 300],
    'classifier__max_depth': [None, 10, 20, 30],
    'classifier__min_samples_split': [2, 5, 10],
    'classifier__min_samples_leaf': [1, 2, 4],
    'classifier__max_features': ['sqrt', 'log2', None]
}

In [ ]:
rf_scaled = build_scaled_classificator(RandomForestClassifier(random_state=42))

cv_stratified = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
rnd_search_rf_1 = RandomizedSearchCV(
    estimator=rf_scaled,
    param_distributions=param_grid,
    n_iter=10,
    cv=cv_stratified,
    scoring="accuracy",
    verbose=1,
    random_state=42,
    n_jobs=-1
)

rnd_search_rf_1.fit(X_train_merged, y_train_merged)

print(f"\nBest params: {rnd_search_rf_1.best_params_}")
print(f"Accuracy in CV: {rnd_search_rf_1.best_score_ * 100:.2f}%")

best_pipeline_model1_rf = rnd_search_rf_1.best_estimator_

In [ ]:
def print_model_heatmap(model, cm, title):
    plt.figure(figsize=(16, 10))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=model.classes_,
                yticklabels=model.classes_)
    plt.title(f"Heatmap of {title} data", fontsize=16)
    plt.xlabel(f"{title} - Predicted", fontsize=12)
    plt.ylabel(f"{title} - Actual", fontsize=12)
    plt.xticks(rotation=45, ha='right')
    plt.show()

In [ ]:
y_pred_rf_merged = best_pipeline_model1_rf.predict(X_test_merged)

print("\n--- REPORT (RANDOM FOREST) ---")
print(classification_report(y_test_merged, y_pred_rf_merged))

cm_rf_1 = confusion_matrix(y_test_merged, y_pred_rf_merged, labels=best_pipeline_model1_rf.classes_)

print_model_heatmap(best_pipeline_model1_rf, cm_rf_1, "Crop")

Riassumendo in una funzione:
Viene creato il modello di classificazione con i parametri selezionati da randomized search, applicando la stratified Kfold.

In [ ]:
def build_model_with_classifier(model, param_grid, X, y):
    cv_stratified = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    rnd_search_m = RandomizedSearchCV(
        estimator=model,
        param_distributions=param_grid,
        n_iter=10,
        cv=cv_stratified,
        scoring="accuracy",
        verbose=1,
        random_state=42,
        n_jobs=-1
    )

    rnd_search_m.fit(X, y)
    return rnd_search_m

In [ ]:
## non serve più???

def train_random_forest(X, y, title) :
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, random_state=42)

    rf_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
    rf_model.fit(X_train, y_train)

    y_pred = rf_model.predict(X_test)

    accuracy = accuracy_score(y_test, y_pred)

    cm = confusion_matrix(y_test, y_pred, labels=rf_model.classes_)

    print(f"Accuracy: {accuracy}")

    plt.figure(figsize=(16, 10))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=rf_model.classes_,
                yticklabels=rf_model.classes_)
    plt.title(f"Heatmap con Random Forest - Predizione {title}")
    plt.xlabel(f"{title} predetto")
    plt.ylabel(f"{title} corretto")
    plt.xticks(rotation=45, ha='right')
    plt.show()

    return rf_model

In [ ]:
# non serve piu???
def BestParams(n_estimators, max_depth, min_samples_split, min_samples_leaf):
    # Parametri da esplorare
    param_grid = {
        'n_estimators': n_estimators,
        'max_depth': max_depth,
        'min_samples_split': min_samples_split,
        'min_samples_leaf': min_samples_leaf
    }

    rf_base = RandomForestClassifier(random_state=42)

    validation_stratified = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    rf_random = RandomizedSearchCV(
        estimator=rf_base,
        param_distributions=param_grid,
        n_iter=20,
        cv=validation_stratified,
        scoring='accuracy',
        verbose=2,
        random_state=42,
        n_jobs=-1
    )

    rf_random.fit(X_merged, y_merged)

    return rf_random.best_params_, rf_random.best_score_

In [ ]:
bestParams, bestScore = BestParams([50, 100, 200, 300], [None, 10, 20, 30, 40], [2, 5, 10], [1, 2, 4])
print(f"Best parameters: {bestParams}")
print(f"Best score: {bestScore * 100:.2f}%")

### XGBoost

XGBoost attende delle etichette *numeriche*, perciò è necessario utilizzare `LabelEncoder`per modificarle, prima che esse vengano passate al modello

In [ ]:
le = LabelEncoder()
y_test_enc = le.fit_transform(y_test_merged)
y_train_enc = le.fit_transform(y_train_merged)


In [ ]:
param_grid_xgb = {
    "classifier__n_estimators": [100, 200, 300],
    "classifier__learning_rate": [0.01, 0.05, 0.1, 0.2],
    "classifier__max_depth": [3, 5, 7, 9],
    "classifier__subsample": [0.8, 0.9, 1.0]
}

xgb_scaled = build_scaled_classificator(XGBClassifier(random_state=42, eval_metric="mlogloss", n_jobs=-1, error_score="raise"))

rnd_search_xgb_1= build_model_with_classifier(xgb_scaled, param_grid_xgb, X_train_merged, y_train_enc)

print(f"\nBest params: {rnd_search_xgb_1.best_params_}")
print(f"Accuracy in CV: {rnd_search_xgb_1.best_score_ * 100:.2f}%")

best_pipeline_model1_xgb = rnd_search_xgb_1.best_estimator_

Per rappresentare con label testuali anzichè numeriche:

In [ ]:
def print_model_heatmap_encoded(model, cm, title, classes):
    plt.figure(figsize=(16, 10))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=classes,
                yticklabels=classes)
    plt.title(f"Heatmap of {title} data", fontsize=16)
    plt.xlabel(f"{title} - Predicted", fontsize=12)
    plt.ylabel(f"{title} - Actual", fontsize=12)
    plt.xticks(rotation=45, ha='right')
    plt.show()

In [ ]:
y_pred_xgb = best_pipeline_model1_xgb.predict(X_test_merged)

cm_xgb = confusion_matrix(y_test_enc, y_pred_xgb, labels=best_pipeline_model1_xgb.classes_)
print_model_heatmap_encoded(best_pipeline_model1_xgb, cm_xgb, "Crop", le.classes_)

### SVM

Laura

### Logistic Regression


In [ ]:
lr_scaled = build_scaled_classificator(LogisticRegression(random_state=42, max_iter=2000))

param_grid_lr = {
    'classifier__C': [0.01, 0.1, 1, 10, 100],
    'classifier__solver': ['lbfgs', 'saga'],
    'classifier__class_weight': [None, 'balanced']
}

rnd_search_lr = build_model_with_classifier(lr_scaled, param_grid_lr, X_train_merged, y_train_enc)

print(f"\nBest params: {rnd_search_lr.best_params_}")
print(f"Accuracy in CV: {rnd_search_lr.best_score_ * 100:.2f}%")

best_pipeline_model1_lr = rnd_search_lr.best_estimator_

In [ ]:
y_pred_lr = best_pipeline_model1_lr.predict(X_test_merged)
cm_lr = confusion_matrix(y_test_merged, y_pred_lr, labels=best_pipeline_model1_lr.classes_)
print_model_heatmap(best_pipeline_model1_lr, cm_lr, "Crop")

## Modello 2

Questo modello dovrà predire la feature `fertilizer` sfruttando il dataset `fertilizer_df`, una volta effettuate alcune necessarie modifiche. 
(da togliere colonna remark)

### Preparazione dei dati

Il secondo modello sfrutterà anche le colonne del dataset che presentano dati di tipo **categorico**. Di conseguenza bisogna ragionare sulla codifica di questi dati in modo da poterli inserire nel modello di predizione. L'approccio standard, seguito anche da noi è quello dell'**One-Hot Encoding**, attraverso il quale il dato viene codificato sfruttando un numero di colonne pari alle categorie, delle quali solo quella corrispondente alla variabile in analisi presenterà valore 1. Per esempio, se nella fase precedente abbiamo trovato che una riga presenta la coltura `rice` allora nel dataset che dovremmo esaminare, tale riga avrà, in corrispondenza della colonna `Crop_rice`, il valore 1, mentre per le altre avrà 0.

In [ ]:
def cat_num(df):
    categorical = df.select_dtypes(include=['object', 'str']).columns
    numeric = df.select_dtypes(include=['number']).columns
    return categorical, numeric

In [ ]:
#fertilizer_df senza colonna "Remark"
fertilizer_df = fertilizer_df.drop(columns=["Remark"])
fertilizer_df_data = fertilizer_df.drop(columns=["Fertilizer"])
fertilizer_column = fertilizer_df["Fertilizer"]

# Divisione tra colonne categoriche e numeriche
categorical, numeric = cat_num(fertilizer_df_data)


In [ ]:
print("fertilizer_df columns:")
print("categorical:\t",categorical)
print("numeric:\t",numeric)

In [ ]:
def build_scaled_encoded_preprocessor(numeric, categorical):
    return ColumnTransformer([
        ("numeric", StandardScaler(), numeric ),
        ("categorical", OneHotEncoder(drop='first', sparse_output=False), categorical)
    ])

In [ ]:
def build_scaled_encoded_classificator(classifier, numeric, categorical):
    return Pipeline([
        ("preprocessor", build_scaled_encoded_preprocessor(numeric, categorical)),
        ("classifier", classifier)
    ])

### Random Forest

In [ ]:
X_train_fertilizer, X_test_fertilizer, y_train_fertilizer, y_test_fertilizer = train_test_split(fertilizer_df_data, fertilizer_column)

In [ ]:
rf_scaled_encoded = build_scaled_encoded_classificator(RandomForestClassifier(random_state=42), numeric, categorical)

rnd_search_rf_2 = build_model_with_classifier(rf_scaled_encoded, param_grid, X_train_fertilizer, y_train_fertilizer)

In [ ]:

print(f"\nBest params: {rnd_search_rf_2.best_params_}")
print(f"Accuracy in CV: {rnd_search_rf_2.best_score_ * 100:.2f}%")

best_pipeline_model2_rf = rnd_search_rf_2.best_estimator_

y_pred_rf_fertilizer = best_pipeline_model2_rf.predict(X_test_fertilizer)
cm_rf_2 = confusion_matrix(y_test_fertilizer, y_pred_rf_fertilizer, labels=best_pipeline_model2_rf.classes_)
print_model_heatmap(best_pipeline_model2_rf, cm_rf_2, "Fertilizer")



### XGBoost


In [ ]:
fert_labels = LabelEncoder()
y_encoded = fert_labels.fit_transform(fertilizer_column)

In [ ]:
X_train_f, X_test_f, y_train_f, y_test_f = train_test_split(fertilizer_df_data, y_encoded, test_size=0.3, random_state=42, stratify=y_encoded)

print(X_train_f)

In [ ]:
categorical_columns, numerical_columns = cat_num(fertilizer_df_data)


preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numerical_columns),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_columns)
])

pipeline_xgb = Pipeline(steps=[
    ("preproc", preprocessor),
    ("classifier", XGBClassifier(random_state=42, eval_metric="mlogloss", n_jobs=-1))
])

param_grid_xgb = {
    "classifier__n_estimators": [100, 200, 300],
    "classifier__learning_rate": [0.01, 0.05, 0.1, 0.2],
    "classifier__max_depth": [3, 5, 7, 9],
    "classifier__subsample": [0.8, 0.9, 1.0]
}


In [ ]:
rnd_search_xgb_2 = build_model_with_classifier(pipeline_xgb, param_grid_xgb, X_train_f, y_train_f)

print(f"\nBest params: {rnd_search_xgb_2.best_params_}")
print(f"Accuracy in CV: {rnd_search_xgb_2.best_score_ * 100:.2f}%")

best_pipeline_model2_xgb = rnd_search_xgb_2.best_estimator_

# Interfaccia Web

In [ ]:
import joblib

rf_crop_model_name = 'rf-crop-model.pkl'
# joblib.dump(rf_model, rf_crop_model_name)